# Lecture 7 Examples and Case — Analysis and Visualisation

**Course:** AAU E26 — Introduction to Scripting, Data Mining and Machine Learning  
**Lecture:** Lecture 7  
**Goal:** Calculate descriptive summaries, make labelled charts, and interpret them cautiously.

[Open this notebook in Google Colab](https://colab.research.google.com/github/asmrabbi/E26_TAN7_Scripting_CPH/blob/main/notebooks/examples/L07_examples_analysis_visualisation.ipynb) · [View the course repository](https://github.com/asmrabbi/E26_TAN7_Scripting_CPH)

Run the cells from top to bottom. Every executable line includes a short comment explaining what it does.


## Goal

Calculate descriptive summaries, compare mean and median, make suitable charts, and separate observation from interpretation.


## Setup

The notebook repeats the transparent minimum cleaning required for analysis.


## Steps

### 1. Load and prepare analysis data


In [1]:
from pathlib import Path  # Imports Path so the notebook can find a local course file when available.
import pandas as pd  # Imports pandas for reading and working with table-shaped data.
remote_data_url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/monthly_service_report.csv"  # Stores the public GitHub address used by Google Colab.
local_data_candidates = [Path("data/monthly_service_report.csv"), Path("../../data/monthly_service_report.csv")]  # Lists possible local paths used during validation.
data_source = next((path for path in local_data_candidates if path.exists()), remote_data_url)  # Chooses a local file when present and otherwise uses GitHub.
analysis_data = pd.read_csv(data_source)  # Reads the CSV file into a pandas DataFrame.
print(analysis_data.head())  # Prints a small preview so we can confirm that loading worked.


   record_id report_month          city service_type  cases_received  \
0       1001   2026-01-01    Copenhagen      Housing             120   
1       1002   2026-01-01   copenhagen     Transport              85   
2       1003   2026-01-01       AALBORG      Housing              -3   
3       1004   2026-02-01    Koebenhavn   Employment              74   
4       1004   2026-02-01    Koebenhavn   Employment              74   

   cases_resolved resolution_days  satisfaction_score  \
0             112             5.1                 4.2   
1              90             3.2                 4.6   
2               0             8.4                 3.1   
3              68             4.0                 4.0   
4              68             4.0                 4.0   

                                  feedback  
0           Helpful staff and clear answer  
1  Quick answer but the form was confusing  
2                  Long wait for an answer  
3                   The guidance was clear  

In [2]:
import matplotlib.pyplot as plt  # Imports plotting tools used for the charts.
analysis_data["city"] = analysis_data["city"].astype("string").str.strip().str.lower().replace({"koebenhavn": "copenhagen"}).str.title()  # Standardises the city labels for grouping.
analysis_data["cases_received"] = pd.to_numeric(analysis_data["cases_received"], errors="coerce")  # Converts received-case values into numbers.
analysis_data["cases_resolved"] = pd.to_numeric(analysis_data["cases_resolved"], errors="coerce")  # Converts resolved-case values into numbers.
analysis_data["resolution_days"] = pd.to_numeric(analysis_data["resolution_days"], errors="coerce")  # Converts resolution times into numbers.
analysis_data["satisfaction_score"] = pd.to_numeric(analysis_data["satisfaction_score"], errors="coerce")  # Converts satisfaction scores into numbers.
analysis_data = analysis_data.drop_duplicates().copy()  # Removes confirmed exact duplicates before summarising.


Matplotlib is building the font cache; this may take a moment.


### 2. Calculate overall summaries


In [3]:
received_count = analysis_data["cases_received"].count()  # Counts non-missing received-case observations.
mean_received = analysis_data["cases_received"].mean()  # Calculates the arithmetic mean.
median_received = analysis_data["cases_received"].median()  # Calculates the middle value after sorting.
print(f"Valid observations: {received_count}")  # Displays the denominator used for the two summaries.
print(f"Mean received: {mean_received:.1f}")  # Displays the mean with one decimal place.
print(f"Median received: {median_received:.1f}")  # Displays the median with one decimal place.


Valid observations: 15
Mean received: 84.5
Median received: 92.0


### 3. Compare mean and median in a skewed example


In [4]:
skewed_wait_times = [2, 3, 3, 4, 30]  # Stores four typical waits and one unusually long wait.
mean_wait = sum(skewed_wait_times) / len(skewed_wait_times)  # Calculates the mean using every value.
sorted_waits = sorted(skewed_wait_times)  # Sorts the list so its middle value is visible.
median_wait = sorted_waits[len(sorted_waits) // 2]  # Selects the middle value for this odd-length list.
print(f"Mean: {mean_wait}; median: {median_wait}")  # Displays how the unusual value affects the mean more strongly.


Mean: 8.4; median: 3


### 4. Group by city


In [5]:
city_summary = analysis_data.groupby("city", dropna=False).agg(records=("record_id", "count"), median_resolution_days=("resolution_days", "median"), mean_satisfaction=("satisfaction_score", "mean")).reset_index()  # Calculates labelled summaries at city level.
print(city_summary.round(2))  # Displays the grouped table with readable rounding.


         city  records  median_resolution_days  mean_satisfaction
0     Aalborg        7                     5.0               3.81
1  Copenhagen        8                     3.8               4.53


### 5. Bar chart for a categorical comparison


In [6]:
service_counts = analysis_data["service_type"].astype("string").str.strip().str.title().value_counts()  # Counts records in each standardised service category.
plt.figure(figsize=(8, 4))  # Creates a figure with a readable classroom aspect ratio.
service_counts.plot(kind="bar", color="#005AA0")  # Draws one bar for each service category.
plt.title("Monthly records by service type")  # Adds a neutral title describing what is plotted.
plt.xlabel("Service type")  # Labels the categorical horizontal axis.
plt.ylabel("Number of records")  # Labels the numerical vertical axis.
plt.xticks(rotation=25, ha="right")  # Rotates long labels so they do not overlap.
plt.tight_layout()  # Adjusts spacing so labels fit inside the figure.
plt.show()  # Displays the completed chart.


/var/folders/1j/v1lnwk6d5xv6pzqk_yc9xjcc0000gn/T/ipykernel_7478/2557576444.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Displays the completed chart.


### 6. Scatter plot for two numeric variables


In [7]:
plot_data = analysis_data.dropna(subset=["resolution_days", "satisfaction_score"])  # Keeps rows that have both values required for the plot.
plt.figure(figsize=(7, 4))  # Creates a new figure for the relationship view.
plt.scatter(plot_data["resolution_days"], plot_data["satisfaction_score"], color="#D1495B", alpha=0.8)  # Draws one point for each complete observation.
plt.title("Resolution time and satisfaction score")  # Describes the variables without claiming causation.
plt.xlabel("Resolution days")  # Labels the horizontal measure.
plt.ylabel("Satisfaction score (1–5)")  # Labels the vertical measure and its intended scale.
plt.grid(alpha=0.2)  # Adds light reference lines without dominating the data.
plt.tight_layout()  # Prevents labels from being cut off.
plt.show()  # Displays the completed scatter plot.


/var/folders/1j/v1lnwk6d5xv6pzqk_yc9xjcc0000gn/T/ipykernel_7478/1819808592.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Displays the completed scatter plot.


### 7. Simple word-frequency summary


In [8]:
import re  # Imports regular-expression tools for transparent tokenisation.
from collections import Counter  # Imports a counter for tallying repeated tokens.
combined_feedback = analysis_data["feedback"].dropna().astype(str).str.cat(sep=" ").lower()  # Combines and lowercases non-missing feedback.
feedback_tokens = re.findall(r"[a-z]+", combined_feedback)  # Extracts alphabetic word tokens.
frequency_table = Counter(feedback_tokens).most_common(10)  # Selects the ten most frequent tokens.
print(frequency_table)  # Displays the bounded word-frequency result.


[('answer', 5), ('the', 5), ('helpful', 4), ('and', 4), ('clear', 4), ('quick', 3), ('was', 3), ('long', 3), ('wait', 3), ('response', 3)]


## Checks

Every table should expose its denominator, and every chart should state variables and units. A visible relationship in the scatter plot does not by itself establish a causal effect.


## Next Steps

Use the exercise notebook to choose a chart based on an analytical question and write one sentence about what the chart shows and one about what it cannot establish.
